In [2]:
import sys
import plotly.express as px
import pandas as pd
from src.data_loader import load_master
from src.data_loader import load_counties

sys.path.insert(0, "..")

#load dataframe
df = load_master()

#make the dataframe for the state and crop inputed
def state_make_df(state, crop):
    state_df = df[
        (df["state_alpha"] == state) &
        (df["crop"] == crop)
    ]
    return state_df 

# Keep counties experiencing drought severity greater than 2
def state_priority_map(state, crop, state_df):
    state_df = state_make_df(state, crop)
    priority_map_df = state_df[
        state_df["max_drought_severity"] > 2
    ]
    #If multiple years exist, aggregate data by county
    priority_map_df = (
        priority_map_df.groupby(["fips", "county_name"], as_index=False)
        .agg(
            max_drought_severity=("max_drought_severity", "mean"),
            yield_anomaly_pct=("yield_anomaly_pct", "mean")
         )
    )
    return priority_map_df

def plot_state_df(state, crop, priority_map_df):
    # Load county boundaries
    counties = load_counties()
    
    # Create the priority map using average yield anomaly
    fig = px.choropleth(
        priority_map_df,
        geojson=counties,
        locations="fips",
        color="yield_anomaly_pct",
        color_continuous_scale="RdYlGn",
        range_color=(-40, 40),
        hover_name="county_name",
        hover_data={
            "max_drought_severity": True,
            "yield_anomaly_pct": ":.2f",
            "fips": False
        },
        title=f"{state} {crop}: Drought-Related Conservation Priorities (2000–2025)",
        labels={
            "yield_anomaly_pct": "Yield Anomaly (%)",
            "max_drought_severity": "Average Drought Severity"
        }
    )
    fig.update_geos(fitbounds="locations", visible=False)
    fig.show()